# Preprocessing in four tiny examples

No datasets, just a few numbers. Each example shows one point.

## 1. Missing values: one gap ruins the result

In [9]:
import numpy as np

ages = np.array([25, 41, np.nan, 58, 64])

print("mean:           ", ages.mean())                  # one gap ruins the result
print("mean, ignoring: ", np.nanmean(ages))             # skip the gap
print("gap filled:     ", np.where(np.isnan(ages), np.nanmedian(ages), ages))
a = np.where(np.isnan(ages), np.nanmedian(ages), ages).copy()
print("mean after fill:", a.mean()) 

mean:            nan
mean, ignoring:  47.0
gap filled:      [25.  41.  49.5 58.  64. ]
mean after fill: 47.5


`nan` spreads through every calculation. Either skip it (`np.nanmean`) or fill it (here with the median).

## 2. Scaling: who is most similar to Anna?

Anna is 25 years old and earns 400 000 NOK. Which of the four others is most similar to her?

In [6]:
anna   = np.array([25, 400_000])        # age, salary

names  = ["Bo", "Cy", "Di", "Eli"]      # Who
others = np.array([[26, 450_000],       # Bo:  1 year older, 50 000 more
                   [60, 405_000],       # Cy:  35 years older, 5 000 more
                   [45, 600_000],       # Di
                   [40, 380_000]])      # Eli

# Distance WITHOUT scaling
raw = np.sqrt(((others - anna) ** 2).sum(axis=1))
print("raw distances:   ", raw.round(0))
print("most similar:    ", names[raw.argmin()])

raw distances:    [ 50000.   5000. 200000.  20000.]
most similar:     Cy


Cy is 35 years older, but the age gap is tiny next to the salary numbers, so Cy "wins". Now scale both columns first (z-score):

In [7]:
everyone = np.vstack([anna, others])
mu, std = everyone.mean(axis=0), everyone.std(axis=0)

anna_s   = (anna - mu) / std
others_s = (others - mu) / std

scaled = np.sqrt(((others_s - anna_s) ** 2).sum(axis=1))
print("scaled distances:", scaled.round(2))
print("most similar:    ", names[scaled.argmin()])

scaled distances: [0.63 2.7  2.94 1.18]
most similar:     Bo


Now **Bo** is the most similar: almost the same age as Anna, and a similar salary. Cy, who is 35 years older, is far away. Before scaling, salary decided everything. After scaling, age and salary both count.

## 3. Why we split: a model can memorise

The labels below are pure coin flips, so there is **nothing** to learn. Watch what a model that memorises does.

In [4]:
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X = rng.normal(size=(1000, 2))
y = rng.integers(0, 2, size=1000)       # random labels

X_train, y_train = X[:700], y[:700]
X_test,  y_test  = X[700:], y[700:]

model = KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train)
print("accuracy on training data:", model.score(X_train, y_train))
print("accuracy on test data:    ", model.score(X_test, y_test))

accuracy on training data: 1.0
accuracy on test data:     0.5433333333333333


On the training data the model is 100% right: it simply memorised every point. On new data it gets about 54%, close to a coin flip, which is the truth, because there was nothing to learn. Had we only looked at the training score, we would have believed the model was perfect.

## 4. Don't leak: the test set must not change the training data

Scale ages to the range 0 to 1 (min-max). The training ages are 20 to 40. One test person is 60.

In [5]:
train = np.array([20, 25, 30, 35, 40])
test  = np.array([60])

# WRONG: min and max from ALL the data
all_data = np.concatenate([train, test])
lo, hi = all_data.min(), all_data.max()
print("leaky   train:", ((train - lo) / (hi - lo)).round(2))

# RIGHT: min and max from the training data only
lo, hi = train.min(), train.max()
print("correct train:", ((train - lo) / (hi - lo)).round(2))
print("correct test: ", ((test  - lo) / (hi - lo)).round(2))

leaky   train: [0.   0.12 0.25 0.38 0.5 ]
correct train: [0.   0.25 0.5  0.75 1.  ]
correct test:  [2.]


In the leaky version the training numbers only reach 0.5 because the test person (60) was included in the min and max: the test set already influenced the training data. In the correct version the training data is scaled on its own (0 to 1), and the test person lands at 2.0, outside the range, exactly as brand-new data would.

## Summary

- **Missing values:** `nan` spoils calculations. Skip or fill the gaps on purpose.
- **Scaling:** without it, the feature with the biggest numbers decides.
- **Splitting:** judge a model on data it has never seen.
- **Leakage:** learn min, max, mean and so on from the training data only, then apply them to the rest.